# Multi-teacher probe (Hướng A) — NuInsSeg committee

**Attach:** `ipateam/nuinsseg` + dataset chứa 2 checkpoint (NuLite-T + LKCell model_best).  
**Settings → Internet: ON** (để clone repo).  
Chạy tuần tự. Cell checkpoint tự dò path. Test 5 ảnh trước rồi mới full.


## 1. Clone repo


In [ ]:
!git clone -q https://github.com/ThuHiep/distill_patho_adaptivepbjcionl.git /kaggle/working/repo
!git clone -q https://github.com/CosmoIknosLab/NuLite.git /kaggle/working/NuLite
!git clone -q https://github.com/ztgu/LKCell.git /kaggle/working/LKCell
print('cloned')


## 2. Dò checkpoint (tự set NULITE / LKCELL)


In [ ]:
import glob
pths = glob.glob('/kaggle/input/**/*.pth', recursive=True) + glob.glob('/kaggle/input/**/*.pt', recursive=True)
NULITE = next((p for p in pths if 'nulite' in p.lower()), None)
LKCELL = next((p for p in pths if any(k in p.lower() for k in ['lkcell','unireplknet','unirepiknet'])), None)
print('NULITE =', NULITE)
print('LKCELL =', LKCELL)
print('--- mọi .pth/.pt ---'); [print(' ', p) for p in pths]
assert NULITE and LKCELL, 'Chưa dò được ckpt — xem list trên, gán tay NULITE=... LKCELL=...'


## 3. Prep ảnh NuInsSeg → PNG + gt_counts.csv


In [ ]:
!cd /kaggle/working/repo/distillation_counting && REPO=/kaggle/working/repo \
  python prep_nuinsseg_as_pannuke.py --out /kaggle/working/repo/work/nuinsseg_png --mode resize
!echo '#ảnh:' $(ls /kaggle/working/repo/work/nuinsseg_png/images | wc -l)
!head -3 /kaggle/working/repo/work/nuinsseg_png/gt_counts.csv


## 4. NuLite — TEST 5 ảnh (bắt lỗi shape/token trước)
Nếu lỗi shape → đổi `--infer_size 256` rồi `1024`. Nếu lỗi `retrieve_tokens` → thêm `--no_tokens`. Nếu ImportError → `!pip install -q <tên>`.


In [ ]:
import subprocess
def run(cmd): print('>>', cmd); return subprocess.run(cmd, shell=True)
IMG='/kaggle/working/repo/work/nuinsseg_png/images'
run(f'cd /kaggle/working/repo/distillation_counting && python dump_cellvit_counts.py --nulite \
  --cellvit_dir /kaggle/working/NuLite --ckpt "{NULITE}" --images_dir {IMG} \
  --out_csv nulite_preds.csv --gpu 0 --mag 40 --infer_size 0 --limit 5')


## 5. NuLite — FULL (chạy khi test OK)


In [ ]:
run(f'cd /kaggle/working/repo/distillation_counting && python dump_cellvit_counts.py --nulite \
  --cellvit_dir /kaggle/working/NuLite --ckpt "{NULITE}" --images_dir {IMG} \
  --out_csv nulite_preds.csv --gpu 0 --mag 40 --infer_size 0')


## 6. LKCell — TEST 5 ảnh


In [ ]:
run(f'cd /kaggle/working/repo/distillation_counting && python dump_cellvit_counts.py --lkcell \
  --cellvit_dir /kaggle/working/LKCell --ckpt "{LKCELL}" --images_dir {IMG} \
  --out_csv lkcell_preds.csv --gpu 0 --mag 40 --infer_size 0 --limit 5')


## 7. LKCell — FULL


In [ ]:
run(f'cd /kaggle/working/repo/distillation_counting && python dump_cellvit_counts.py --lkcell \
  --cellvit_dir /kaggle/working/LKCell --ckpt "{LKCELL}" --images_dir {IMG} \
  --out_csv lkcell_preds.csv --gpu 0 --mag 40 --infer_size 0')


## 8. PROBE 4-teacher (PathoSAM + SAM3 + NuLite + LKCell)
Đo consensus-MAE vs teacher đơn + corr(std-disagreement, |lỗi|). Cần cell 5 & 7 xong.


In [ ]:
!cd /kaggle/working/repo/distillation_counting && REPO=/kaggle/working/repo \
  python probe_multiteacher_full.py


## 9. (tùy) Lưu CSV để backup
Tải `nulite_preds.csv`, `lkcell_preds.csv` ở `/kaggle/working/repo/distillation_counting/` về máy / upload dataset.


In [ ]:
!cp /kaggle/working/repo/distillation_counting/nulite_preds.csv /kaggle/working/ 2>/dev/null
!cp /kaggle/working/repo/distillation_counting/lkcell_preds.csv /kaggle/working/ 2>/dev/null
!cp /kaggle/working/repo/work/nuinsseg_png/gt_counts.csv /kaggle/working/ 2>/dev/null
print('copied to /kaggle/working (tải ở panel Output)')
